# 03 · Gold · `fato_troca_medidor`
### V-Energy Grid 360 — Arquitetura Medalhão

As 6.299 trocas de medidor se concentram entre `2025-09` e `2026-03`, dentro dos últimos 12 meses do diagnóstico. Esta tabela permite **testar se a troca mudou o volume medido** (P1) e **mostrar a troca na visão 360** quando o cliente reclama de conta alta (P5, Camila). Grava `v_energy.gold.fato_troca_medidor`, seguindo o **contrato da Gold (tabela A6, versão v4)**, com as 13 colunas do contrato.

**Grão:** 1 linha = 1 troca de medidor. Esperado: **6.299 linhas** (cada UC trocou o medidor uma vez).

**Fontes (Silver):** `leituras` (o mês da troca e o consumo antes e depois), `medidores` (tipo, instalação e retirada) e `unidades_consumidoras` (classe, município e bairro).

| # | Passo | Regra |
|---|---|---|
| 1 | Troca | linhas de `silver.leituras` com `troca_medidor = true` (log L5): o mês da troca e o **medidor novo** (o que fez a leitura) |
| 2 | Medidor antigo | o medidor da leitura do **mês anterior** da mesma UC |
| 3 | Tipos e idade | de `silver.medidores`; idade do antigo = da instalação até a **retirada** |
| 4 | Consumo antes e depois | média do `consumo_kwh` dos **6 meses anteriores** ao mês da troca e dos **até 6 meses seguintes** (o mês da troca fica de fora dos dois) |
| 5 | Localização | `classe`, `municipio` e `bairro` de `silver.unidades_consumidoras` |

**Decisões deste notebook:**
- **O mês da troca não entra em nenhuma das médias.** Nele, o consumo é a `leitura_atual` do medidor novo (log L5), e o preço por kWh fica ~5% acima do típico: é um mês de critério diferente.
- **"Até 6 meses depois":** a base termina em `2026-06`, então as trocas de `2026-01` a `2026-03` têm de 5 a 3 meses depois. A média usa os meses que existem.
- **Cuidado com a estação do ano (não é decisão, é leitura):** as trocas são de setembro a março. Os 6 meses antes caem no inverno (consumo baixo) e os 6 depois no verão (consumo alto), então a `variacao_consumo_pct` **sobe mesmo sem efeito da troca**. A seção 9 compara com as UCs que **não** trocaram, nas mesmas janelas, para separar o efeito da troca do efeito da estação.

**Pré-requisito:** `silver.leituras`, `silver.medidores` e `silver.unidades_consumidoras` gravadas. Para o teste de conexão, a `gold.dim_uc`.

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from pyspark.sql import functions as F, Row, Window
from datetime import datetime

tabela_leit = f"{silver_schema}.leituras"
tabela_med = f"{silver_schema}.medidores"
tabela_uc = f"{silver_schema}.unidades_consumidoras"
tabela_gold = f"{gold_schema}.fato_troca_medidor"
tabela_dim = f"{gold_schema}.dim_uc"                 # só para o teste de conexão (seção 9)
tabela_dq_log = f"{silver_schema}.dq_log"            # mesmo log das Silvers e das outras Golds

TROCA_INI, TROCA_FIM = "2025-09", "2026-03"          # período das trocas (log L5)
JANELA = 6                                           # meses antes e depois

def n_br(n):
    return f"{n:,}".replace(",", ".")

def mes_data(col):
    """'yyyy-MM' -> data do primeiro dia do mês"""
    return F.to_date(F.concat(col, F.lit("-01")))

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(f"origem:  {tabela_leit}, {tabela_med}, {tabela_uc}")
print(f"destino: {tabela_gold}")

## 2. Funções de checagem de qualidade

Mesmas funções das Silvers e das outras Golds, com as mesmas colunas no log. Os resultados vão para `silver.dq_log` (sempre), e a tabela só é gravada se **todas** as checagens passarem.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=failed, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=dupes, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=diferenca, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

def dq_check_valor(table_name: str, check_name: str, total_rows: int, obtido, esperado):
    """Compara um valor calculado com o esperado."""
    passed = obtido == esperado
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total_rows,
                          failed_rows=0 if passed else 1, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | obtido {obtido} (esperado {esperado})")

## 3. Leitura das Silvers e conferência das colunas

Contagens esperadas: **2.152.336** leituras, das quais **6.299** com `troca_medidor = true`; **51.299** medidores (45.000 ativos e 6.299 retirados); **45.000** UCs.

In [0]:
leit = spark.table(tabela_leit)
med = spark.table(tabela_med)
uc = spark.table(tabela_uc)

exigidas = {
    tabela_leit: ["id_uc", "mes_referencia", "id_medidor", "consumo_kwh", "troca_medidor"],
    tabela_med: ["id_medidor", "id_uc", "tipo", "status", "data_instalacao", "data_retirada"],
    tabela_uc: ["id_uc", "classe", "municipio", "bairro"],
}
for tabela, colunas in exigidas.items():
    faltando = [c for c in colunas if c not in spark.table(tabela).columns]
    if faltando:
        raise Exception(f"{tabela} não tem as colunas {faltando} (DESCRIBE {tabela}).")

n_trocas_leit = leit.filter("troca_medidor").count()
print(f"leituras: {n_br(leit.count())} | com troca: {n_br(n_trocas_leit)}")   # esperado: 2.152.336 | 6.299
med.groupBy("status").count().show()                                       # esperado: ATIVO 45.000 | RETIRADO 6.299
print("UCs:", n_br(uc.count()))                                            # esperado: 45.000

## 4. As trocas: mês, medidor novo e medidor antigo

**Medidor novo:** o que fez a leitura no mês da troca (`id_medidor` da linha com `troca_medidor = true`).

**Medidor antigo:** o que fez a leitura do **mês anterior** da mesma UC (`lag` por UC, ordenado por mês). O log L5 confirma que o antigo está `RETIRADO`, com `data_retirada` no mês da troca, em 100% dos casos; a seção 7 confere de novo.

**Conferência:** 6.299 trocas, nenhuma sem medidor antigo, e o antigo diferente do novo.

In [0]:
w_uc = Window.partitionBy("id_uc").orderBy("mes_referencia")
leit_seq = (leit.select("id_uc", "mes_referencia", "id_medidor", "troca_medidor")
            .withColumn("id_medidor_mes_anterior", F.lag("id_medidor").over(w_uc)))

trocas = (leit_seq.filter("troca_medidor")
          .select("id_uc", F.col("mes_referencia").alias("mes_troca"),
                  F.col("id_medidor_mes_anterior").alias("id_medidor_antigo"),
                  F.col("id_medidor").alias("id_medidor_novo")))

print("trocas:", n_br(trocas.count()))                                                        # esperado: 6.299
print("sem medidor antigo:", trocas.filter("id_medidor_antigo IS NULL").count())               # esperado: 0
print("antigo igual ao novo:", trocas.filter("id_medidor_antigo = id_medidor_novo").count())   # esperado: 0
trocas.groupBy("mes_troca").count().orderBy("mes_troca").show()                                 # esperado: ~900 por mês, de 2025-09 a 2026-03

## 5. Tipos dos medidores e idade do antigo

- **Tipo:** `tipo` de `silver.medidores`, para o antigo e para o novo.
- **Idade do antigo na retirada:** dias entre `data_instalacao` e `data_retirada`, ÷ 365,25, com 1 casa (a mesma conta da `idade_medidor_anos` da `dim_uc`, mas até a retirada, e não até o fim da base).

In [0]:
m_antigo = med.select(F.col("id_medidor").alias("id_medidor_antigo"), F.col("id_uc").alias("uc_antigo"),
                     F.col("tipo").alias("tipo_medidor_antigo"), F.col("status").alias("status_antigo"),
                     F.col("data_instalacao").alias("inst_antigo"), F.col("data_retirada").alias("ret_antigo"))
m_novo = med.select(F.col("id_medidor").alias("id_medidor_novo"), F.col("id_uc").alias("uc_novo"),
                    F.col("tipo").alias("tipo_medidor_novo"), F.col("status").alias("status_novo"),
                    F.col("data_instalacao").alias("inst_novo"))

trocas_med = (trocas.join(m_antigo, "id_medidor_antigo", "left").join(m_novo, "id_medidor_novo", "left")
              .withColumn("idade_medidor_antigo_anos",
                          F.round(F.datediff("ret_antigo", "inst_antigo") / 365.25, 1)))
trocas_med.groupBy("tipo_medidor_antigo", "tipo_medidor_novo").count().orderBy(F.desc("count")).show()

## 6. Consumo antes e depois e montagem da tabela

Para cada troca, a média do `consumo_kwh` da mesma UC:
- **antes:** os 6 meses anteriores ao mês da troca (`mes_troca − 6` a `mes_troca − 1`);
- **depois:** os até 6 meses seguintes (`mes_troca + 1` a `mes_troca + 6`, limitado a `2026-06`).

`variacao_consumo_pct = (depois ÷ antes − 1) × 100`, com 2 casas; nula se a média de antes for zero (`try_divide`).

**Ordem das colunas:** a do contrato (13 colunas). As colunas auxiliares (status, datas e UC de cada medidor) ficam só para as checagens.

In [0]:
consumo = leit.select("id_uc", F.col("mes_referencia").alias("mes"), "consumo_kwh")

janela = (trocas.select("id_uc", "mes_troca").join(consumo, "id_uc")
          .withColumn("dist", F.months_between(mes_data(F.col("mes")), mes_data(F.col("mes_troca"))).cast("int"))
          .groupBy("id_uc", "mes_troca").agg(
              F.avg(F.when(F.col("dist").between(-JANELA, -1), F.col("consumo_kwh"))).alias("antes"),
              F.avg(F.when(F.col("dist").between(1, JANELA), F.col("consumo_kwh"))).alias("depois"),
              F.count(F.when(F.col("dist").between(-JANELA, -1), 1)).alias("meses_antes"),
              F.count(F.when(F.col("dist").between(1, JANELA), 1)).alias("meses_depois")))

base = (trocas_med.join(janela, ["id_uc", "mes_troca"], "left")
        .join(F.broadcast(uc.select("id_uc", "classe", "municipio", "bairro")), "id_uc", "left")
        .withColumn("consumo_medio_6m_antes", F.round("antes", 2))
        .withColumn("consumo_medio_6m_depois", F.round("depois", 2))
        .withColumn("variacao_consumo_pct", F.round(F.expr("try_divide(depois, antes) - 1") * 100, 2)))

df_troca = base.select(
    "id_uc", "mes_troca",
    "id_medidor_antigo", "tipo_medidor_antigo",
    F.col("idade_medidor_antigo_anos").cast("decimal(5,1)").alias("idade_medidor_antigo_anos"),
    "id_medidor_novo", "tipo_medidor_novo",
    F.col("consumo_medio_6m_antes").cast("decimal(12,2)").alias("consumo_medio_6m_antes"),
    F.col("consumo_medio_6m_depois").cast("decimal(12,2)").alias("consumo_medio_6m_depois"),
    F.col("variacao_consumo_pct").cast("decimal(8,2)").alias("variacao_consumo_pct"),
    "classe", "municipio", "bairro",
)
print("colunas:", len(df_troca.columns))   # esperado: 13
df_troca.printSchema()
base.groupBy("meses_antes", "meses_depois").count().orderBy("meses_antes", "meses_depois").show()   # esperado: 6 antes; 3 a 6 depois

## 7. Checagens de qualidade

| Checagem | Protege |
|---|---|
| 6.299 linhas; `id_uc` único; igual às leituras com `troca_medidor` | o grão (1 troca por UC) |
| Colunas não nulas | junções |
| `mes_troca` entre `2025-09` e `2026-03` | o período das trocas (L5) |
| Antigo ≠ novo; os dois da mesma UC da troca | a identificação dos medidores |
| **Antigo `RETIRADO`, com `data_retirada` no mês da troca** | a confirmação do cadastro (L5) |
| **Novo `ATIVO`, instalado no mês da troca** | idem (M4, teste 3) |
| Tipos nos 3 valores; idade do antigo >= 0 | domínios |
| 6 meses antes em todas; de 3 a 6 meses depois | as janelas |
| Consumos >= 0; variação coerente com as médias | os cálculos |

In [0]:
t = "gold.fato_troca_medidor"
dq_results.clear()

TIPOS = ["ELETRONICO", "ELETROMECANICO", "SMART"]
def ok(cond):
    return F.coalesce(cond, F.lit(False))   # nulo também falha

g = df_troca
n_g = g.count()

# O grão
dq_check_contagem(t, "total de trocas = 6.299", g, 6_299)
dq_check_unique(t, "id_uc único (1 troca por UC)", g, ["id_uc"])
dq_check_valor(t, "trocas = leituras com troca_medidor", n_g, n_g, n_trocas_leit)

for coluna in df_troca.columns:
    dq_check(t, f"{coluna} não nulo", g, F.col(coluna).isNotNull())

# Período e medidores
dq_check(t, f"mes_troca entre {TROCA_INI} e {TROCA_FIM}", g, ok(F.col("mes_troca").between(TROCA_INI, TROCA_FIM)))
dq_check(t, "medidor antigo diferente do novo", g, ok(F.col("id_medidor_antigo") != F.col("id_medidor_novo")))
dq_check(t, "antigo e novo pertencem à UC da troca", base,
         ok((F.col("uc_antigo") == F.col("id_uc")) & (F.col("uc_novo") == F.col("id_uc"))))
dq_check(t, "antigo RETIRADO, com data_retirada no mês da troca", base,
         ok((F.col("status_antigo") == "RETIRADO") & (F.date_format("ret_antigo", "yyyy-MM") == F.col("mes_troca"))))
dq_check(t, "novo ATIVO, instalado no mês da troca", base,
         ok((F.col("status_novo") == "ATIVO") & (F.date_format("inst_novo", "yyyy-MM") == F.col("mes_troca"))))
dq_check(t, "tipos dos medidores nos 3 valores", g,
         ok(F.col("tipo_medidor_antigo").isin(*TIPOS) & F.col("tipo_medidor_novo").isin(*TIPOS)))
dq_check(t, "idade do antigo na retirada >= 0", g, ok(F.col("idade_medidor_antigo_anos") >= 0))

# Janelas e cálculos
# 6 meses antes; menos só nas UCs ligadas há menos de 6 meses (nelas, todos os meses desde a 1ª leitura)
primeiro = leit.groupBy("id_uc").agg(F.min("mes_referencia").alias("primeiro_mes"))
esperado_antes = F.least(F.lit(JANELA), F.months_between(mes_data(F.col("mes_troca")), mes_data(F.col("primeiro_mes"))).cast("int"))
dq_check(t, "meses antes = 6, ou todos desde a 1ª leitura (UC nova)", base.join(primeiro, "id_uc"), F.col("meses_antes") == esperado_antes)
dq_check(t, "de 3 a 6 meses depois", base, F.col("meses_depois").between(3, JANELA))
dq_check(t, "consumos médios >= 0", g, ok((F.col("consumo_medio_6m_antes") >= 0) & (F.col("consumo_medio_6m_depois") >= 0)))
dq_check(t, "variação coerente com as médias (tolerância de 0,1 p.p.)", g.filter("consumo_medio_6m_antes > 0"),
         ok(F.abs(F.col("variacao_consumo_pct")
                  - (F.col("consumo_medio_6m_depois") / F.col("consumo_medio_6m_antes") - 1) * 100) <= 0.1))
dq_check_valor(t, "13 colunas, como no contrato", n_g, len(df_troca.columns), 13)

## 8. Registro das checagens e gravação, só se todas passarem

O log vai **sempre** para `silver.dq_log` (`table_name = gold.fato_troca_medidor`); a tabela só é gravada se **todas** as checagens passarem.

In [0]:
spark.createDataFrame(dq_results).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(tabela_dq_log)

falhas = [r for r in dq_results if not r.passed]
if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_gold} NÃO foi gravada:")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_gold} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

df_troca.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_gold)
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_gold} gravada: {n_br(spark.table(tabela_gold).count())} linhas")

## 9. Teste de conexão e consultas de conferência

1. **Conexão com a `dim_uc`:** toda UC da tabela tem `medidor_trocado_recentemente = true`, e o medidor novo é o `id_medidor_ativo` da dimensão.
2. **Quem trocou o quê:** tipo antigo → novo, idade do antigo, trocas por mês e por classe.
3. **A troca mudou o volume medido? (P1)** A variação das UCs que trocaram **comparada com a das UCs que não trocaram**, nas mesmas janelas de cada mês de troca. A diferença entre as duas é o efeito da troca, sem o efeito da estação do ano. As UCs com GD conectada ficam de fora dos dois grupos (a GD derruba o consumo e confundiria a comparação).
4. **Visão 360 (P5):** como a troca aparece para o atendente.

In [0]:
f = spark.table(tabela_gold)

# 1. Conexão com a dim_uc (só se ela existir neste workspace)
if spark.catalog.tableExists(tabela_dim):
    d = spark.table(tabela_dim).select("id_uc", "id_medidor_ativo", "medidor_trocado_recentemente", "possui_gd", "data_conexao_gd")
    c = f.join(d, "id_uc", "left")
    print("UCs da troca fora da gold.dim_uc:", n_br(c.filter("id_medidor_ativo IS NULL").count()))                        # esperado: 0
    print("sem medidor_trocado_recentemente na dim_uc:", n_br(c.filter("NOT medidor_trocado_recentemente").count()))      # esperado: 0
    print("medidor novo diferente do ativo da dim_uc:", n_br(c.filter("id_medidor_novo <> id_medidor_ativo").count()))    # esperado: 0
    print("UCs com troca em toda a dim_uc:", n_br(d.filter("medidor_trocado_recentemente").count()))                      # esperado: 6.299
else:
    print(f"{tabela_dim} não existe neste workspace: teste de conexão fica para quem tiver as duas tabelas")

# 2. Quem trocou o quê
display(f.groupBy("tipo_medidor_antigo", "tipo_medidor_novo").agg(F.count("*").alias("trocas"),
        F.round(F.avg("idade_medidor_antigo_anos"), 1).alias("idade_media_antigo")).orderBy(F.desc("trocas")))
display(f.groupBy("classe").count().orderBy(F.desc("count")))

In [0]:
# 3. A troca mudou o volume medido? Comparação com as UCs que NÃO trocaram, nas mesmas janelas
#    (para cada mês de troca m: antes = m-6 a m-1, depois = m+1 a m+6). Sem UCs com GD nos dois grupos.
ucs_gd = spark.table(tabela_dim).filter("possui_gd").select("id_uc") if spark.catalog.tableExists(tabela_dim) \
         else spark.table(f"{silver_schema}.geracao_distribuida").select("id_uc")
meses_troca = f.select("mes_troca").distinct()

controle = (consumo.filter(F.col("mes").between("2025-03", "2026-06"))     # só os meses das janelas (deixa a conta leve)
            .join(f.select("id_uc"), "id_uc", "left_anti").join(ucs_gd, "id_uc", "left_anti")
            .crossJoin(meses_troca)
            .withColumn("dist", F.months_between(mes_data(F.col("mes")), mes_data(F.col("mes_troca"))).cast("int"))
            .groupBy("mes_troca").agg(
                F.avg(F.when(F.col("dist").between(-JANELA, -1), F.col("consumo_kwh"))).alias("antes_ctrl"),
                F.avg(F.when(F.col("dist").between(1, JANELA), F.col("consumo_kwh"))).alias("depois_ctrl"))
            .withColumn("variacao_ctrl_pct", F.round((F.col("depois_ctrl") / F.col("antes_ctrl") - 1) * 100, 1)))

trocou = (f.join(ucs_gd, "id_uc", "left_anti").groupBy("mes_troca").agg(
              F.count("*").alias("trocas"),
              F.avg("consumo_medio_6m_antes").alias("antes_troca"), F.avg("consumo_medio_6m_depois").alias("depois_troca"))
          .withColumn("variacao_troca_pct", F.round((F.col("depois_troca") / F.col("antes_troca") - 1) * 100, 1)))

comp = (trocou.join(controle, "mes_troca")
        .withColumn("efeito_troca_pp", F.round(F.col("variacao_troca_pct") - F.col("variacao_ctrl_pct"), 1))
        .orderBy("mes_troca"))
print("Variação do consumo: UCs que trocaram x UCs que não trocaram (mesmas janelas; sem GD)")
display(comp.select("mes_troca", "trocas", "variacao_troca_pct", "variacao_ctrl_pct", "efeito_troca_pp"))

# Resumo: efeito médio ponderado pelo número de trocas
r = comp.agg((F.sum(F.col("efeito_troca_pp") * F.col("trocas")) / F.sum("trocas")).alias("efeito")).first()["efeito"]
print(f"efeito médio da troca, descontada a estação: {r:+.1f} pontos percentuais")
# Perto de 0: a troca não mudou o volume medido. Muito positivo: o medidor antigo media a menos.

# Distribuição da variação UC a UC (sem GD), para ver se há um grupo que mudou muito
display(f.join(ucs_gd, "id_uc", "left_anti")
        .agg(F.expr("percentile_approx(cast(variacao_consumo_pct as double), array(0.1, 0.25, 0.5, 0.75, 0.9))").alias("p10_p25_p50_p75_p90")))

In [0]:
# 4. Visão 360: como a troca aparece para o atendente (uma UC de exemplo)
ex = f.orderBy("mes_troca", "id_uc").first()
print("UC de exemplo:", ex["id_uc"], "| troca em", ex["mes_troca"], "|", ex["tipo_medidor_antigo"], "->", ex["tipo_medidor_novo"],
      f"| antigo com {ex['idade_medidor_antigo_anos']} anos")
display(consumo.filter(F.col("id_uc") == ex["id_uc"]).filter(F.col("mes").between("2025-01", "2026-06")).orderBy("mes"))